In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor, export_text
import seaborn as sns


# Insert warehouse name here
warehouse_name = "OE"

# GitHub repo
base = "Lucas_Systems_Capstone_Project"

# Load your processed data
df = pd.read_parquet(f"~/{base}/data/processed/oe_detailed_withAO.parquet")

In [2]:
df

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID,Prev_Timestamp,Prev_LocationID,...,Prev_Bay,Prev_Level,Prev_Slot,Aisle2,Bay2,Prev_Aisle2,Prev_Bay2,LocKey,PrevLocKey,Travel_Distance
0,PickPut,143,30,7717848,49658,160,2025-09-08 12:11:50.830,35192,NaT,<NA>,...,<NA>,<NA>,<NA>,40,19,NaN,NaN,40|19|||,NaN,NaN
1,PickPut,143,30,7717860,460,50,2025-09-08 12:12:18.127,422,2025-09-08 12:11:50.830,35192,...,19,2,2,40,18,40,19,40|18|||,40|19|||,21.0
2,PickPut,143,30,7717908,460,100,2025-09-08 12:15:46.650,422,2025-09-08 12:12:18.127,422,...,18,2,1,40,18,40,18,40|18|||,40|18|||,0.0
3,PickPut,143,30,7717921,44547,13,2025-09-08 12:16:30.470,10743,2025-09-08 12:15:46.650,422,...,18,2,1,40,18,40,18,40|18|||,40|18|||,0.0
4,PickPut,143,30,7717920,44547,13,2025-09-08 12:18:00.970,10743,2025-09-08 12:16:30.470,10743,...,18,2,2,40,18,40,18,40|18|||,40|18|||,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96126,PickPut,90,10,8039505,42582,1,2025-12-02 20:42:19.333,213005,2025-12-02 20:41:27.763,388355,...,3,4,4,83,03,79,03,83|03|||,79|03|||,40.0
96127,AssignmentOpen,90,10,8038545,<NA>,<NA>,2025-12-02 20:54:42.537,<NA>,2025-12-02 20:42:19.333,213005,...,3,1,1,NaN,NaN,83,03,NaN,83|03|||,NaN
96128,PickPut,90,30,8039941,52330,4,2025-12-02 22:48:36.767,4710513,2025-12-02 20:54:42.537,<NA>,...,<NA>,<NA>,<NA>,26,06,NaN,NaN,26|06|||,NaN,NaN
96129,PickPut,90,30,8039942,52330,6,2025-12-02 22:48:36.973,4710513,2025-12-02 22:48:36.767,4710513,...,6,1,1,26,06,26,06,26|06|||,26|06|||,0.0


In [3]:
user143_sep08 = df[(df["UserID"] == "90") & (df["Timestamp"].dt.date == pd.to_datetime("2025-09-08").date())]
user143_sep08["AssignmentID"].value_counts()

AssignmentID
7719955    1
7719956    1
7719986    1
7719987    1
7719985    1
7719298    1
7719389    1
7719415    1
7719419    1
7719249    1
7719440    1
7719355    1
7719262    1
7719307    1
7719348    1
7719491    1
7719529    1
7719725    1
7719578    1
7719758    1
7719781    1
7719537    1
7719773    1
7719611    1
7719749    1
7719536    1
7719636    1
7719788    1
7719637    1
7719494    1
7719771    1
7719659    1
7719597    1
7720073    1
7720085    1
7720079    1
7720415    1
7720348    1
7720402    1
7721311    1
7720420    1
7720982    1
7720983    1
7720994    1
7720346    1
7720427    1
7720408    1
Name: count, dtype: int64

In [4]:
user143_sep08["ActivityCode"].value_counts()

ActivityCode
PickPut    47
Name: count, dtype: int64

In [5]:
len(pd.unique(user143_sep08["AssignmentID"]))

47

In [15]:
df_raw = pd.read_csv("../data/database_backups_csv/OE/OE_Activity.csv", header=None, names=["ActivityCode","UserID","WorkCode","AssignmentID","ProductID","Quantity","Timestamp","LocationID"])
df_raw["Timestamp"] = pd.to_datetime(df_raw["Timestamp"])
df_raw["AssignmentID"] = df_raw["AssignmentID"].astype(str)
df_raw

,ActivityCode,UserID,WorkCode,AssignmentID,ProductID,Quantity,Timestamp,LocationID
0,PickPut,419,20,7954566,4289.0,1.0,2025-11-10 11:37:14.160,826367.0
1,AssignmentOpen,64,10,7954429,NaN,NaN,2025-11-10 11:38:34.043,NaN
2,PickPut,419,20,7954541,6592.0,1.0,2025-11-10 11:39:42.330,14524.0
3,PickPut,419,20,7954542,6592.0,1.0,2025-11-10 11:39:42.883,14524.0
4,PickPut,419,20,7954543,6592.0,1.0,2025-11-10 11:39:43.483,14524.0
...,...,...,...,...,...,...,...,...
96127,PickPut,461,30,7726758,52333.0,30.0,2025-09-09 20:20:53.250,97964.0
96128,PickPut,461,30,7726671,12034.0,1.0,2025-09-09 20:21:59.743,77378.0
96129,PickPut,461,30,7726574,2371.0,3.0,2025-09-09 20:22:53.123,53179.0
96130,PickPut,461,30,7726758,52250.0,50.0,2025-09-09 20:24:19.220,25027.0


In [16]:
user90_sep08_raw = df_raw[(df_raw["UserID"] == 90) & (df_raw["Timestamp"].dt.date == pd.to_datetime("2025-09-08").date())]
user90_sep08_raw["AssignmentID"].value_counts()

AssignmentID
7719955    1
7719956    1
7719986    1
7719987    1
7719985    1
7719298    1
7719389    1
7719415    1
7719419    1
7719249    1
7719440    1
7719355    1
7719262    1
7719307    1
7719348    1
7719491    1
7719529    1
7719725    1
7719578    1
7719758    1
7719781    1
7719537    1
7719773    1
7719611    1
7719749    1
7719536    1
7719636    1
7719788    1
7719637    1
7719494    1
7719771    1
7719659    1
7719597    1
7720073    1
7720085    1
7720079    1
7720415    1
7720348    1
7720402    1
7721311    1
7720420    1
7720982    1
7720983    1
7720994    1
7720346    1
7720427    1
7720408    1
Name: count, dtype: int64

In [17]:
sep08_raw = df_raw[df_raw["Timestamp"].dt.date == pd.to_datetime("2025-09-08").date()]
user501_sep08_raw = sep08_raw[sep08_raw["UserID"] == 501]
user501_sep08_raw["WorkCode"].value_counts()

WorkCode
20    983
30     30
10      6
Name: count, dtype: int64

In [18]:
user501_sep08_raw["ActivityCode"].value_counts()

ActivityCode
PickPut           1017
AssignmentOpen       2
Name: count, dtype: int64

In [19]:
user414_sep08_raw = sep08_raw[sep08_raw["UserID"] == 414]
user414_sep08_raw["ActivityCode"].value_counts()

ActivityCode
PickPut    807
Name: count, dtype: int64

In [20]:
sep08_raw["UserID"].value_counts(ascending=True)

UserID
90       47
496     110
412     113
499     204
322     225
390     228
497     286
435     290
419     320
476     324
475     358
348     374
439     388
217     401
440     514
221     529
479     558
488     602
143     622
67      627
484     653
467     708
480     726
471     730
494     748
460     764
262     768
414     807
501    1019
Name: count, dtype: int64

In [21]:
user496_sep08_raw = sep08_raw[sep08_raw["UserID"] == 496]
user496_sep08_raw["AssignmentID"].value_counts()

AssignmentID
7720725    6
7722105    4
7720688    4
7720743    4
7722009    3
          ..
7720545    1
7720546    1
7720620    1
7720726    1
7720727    1
Name: count, Length: 66, dtype: int64

In [29]:
user496_sep08_raw["ActivityCode"].value_counts()

ActivityCode
PickPut    110
Name: count, dtype: int64

In [36]:
user496_start_end = user496_sep08_raw.groupby("AssignmentID").agg({"Timestamp": [np.min, np.max]})
user496_start_end.columns = user496_start_end.columns.droplevel(0)
user496_start_end = user496_start_end.sort_values("min", ascending=True).iloc[0:30].reset_index()
user496_start_end["max"] = user496_start_end["max"] + pd.to_timedelta(0.5, unit='m')
user496_start_end

,AssignmentID,min,max
0,7719691,2025-09-08 16:14:52.123,2025-09-08 16:15:54.237
1,7719692,2025-09-08 16:15:46.937,2025-09-08 16:16:16.937
2,7719682,2025-09-08 16:16:44.560,2025-09-08 16:19:35.190
3,7719683,2025-09-08 16:17:40.607,2025-09-08 16:18:10.607
4,7719708,2025-09-08 16:18:18.360,2025-09-08 16:20:15.830
5,7719631,2025-09-08 16:23:47.913,2025-09-08 16:24:53.497
6,7719731,2025-09-08 16:25:21.973,2025-09-08 16:27:37.743
7,7719732,2025-09-08 16:27:09.840,2025-09-08 16:27:39.840
8,7719635,2025-09-08 16:27:36.013,2025-09-08 16:28:38.960
9,7719632,2025-09-08 16:29:28.743,2025-09-08 16:29:58.743


In [ ]:
import plotly.express as px
fig = px.timeline(user496_start_end, x_start="min", x_end="max", y="AssignmentID")
fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up
fig.update_layout(title_text="User496 9/8/2025 First 30 UoW Gantt Chart")
fig.show()

In [59]:
import plotly.express as px

def plot_gantt(user_id:int, date:str, num_tasks:int=30, jitter:float=0.5):
    df_filtered = df_raw[(df_raw["UserID"] == user_id) & (df_raw["Timestamp"].dt.date == pd.to_datetime(date).date())]
    df_start_end = df_filtered.groupby(["AssignmentID", "ActivityCode"]).agg({"Timestamp": [np.min, np.max]})
    df_start_end.columns = df_start_end.columns.droplevel(0)
    df_start_end = df_start_end.sort_values("min", ascending=True).iloc[0:num_tasks].reset_index()
    df_start_end["max"] = df_start_end["max"] + pd.to_timedelta(jitter, unit='m')
    fig = px.timeline(df_start_end, x_start="min", x_end="max", y="AssignmentID", color="ActivityCode", color_discrete_sequence=["red", "blue"])
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(title_text="User" + f"{user_id}" +  " " + f"{date}" +  " " + f"First {num_tasks} UoW Gantt Chart")
    fig.show()

In [63]:
plot_gantt(496, "2025-09-08", 30)

In [51]:
df_raw[(df_raw["UserID"] == 461) & (df_raw["Timestamp"].dt.date == pd.to_datetime("2025-09-09").date())]["ActivityCode"].value_counts()

ActivityCode
PickPut           166
AssignmentOpen      2
Name: count, dtype: int64

In [56]:
user496_start_end = user496_sep08_raw.groupby(["AssignmentID", "ActivityCode"]).agg({"Timestamp": [np.min, np.max]})
user496_start_end

Timestamp                        
                                              min                     max
AssignmentID ActivityCode                                                
7719631      PickPut      2025-09-08 16:23:47.913 2025-09-08 16:24:23.497
7719632      PickPut      2025-09-08 16:29:28.743 2025-09-08 16:29:28.743
7719635      PickPut      2025-09-08 16:27:36.013 2025-09-08 16:28:08.960
7719682      PickPut      2025-09-08 16:16:44.560 2025-09-08 16:19:05.190
7719683      PickPut      2025-09-08 16:17:40.607 2025-09-08 16:17:40.607
...                                           ...                     ...
7722106      PickPut      2025-09-08 16:51:25.033 2025-09-08 16:51:25.033
7722107      PickPut      2025-09-08 16:52:04.920 2025-09-08 16:52:04.920
7722108      PickPut      2025-09-08 16:52:16.777 2025-09-08 16:52:16.777
7722109      PickPut      2025-09-08 16:53:11.530 2025-09-08 16:53:11.530
7722120      PickPut      2025-09-08 16:54:04.460 2025-09-08 16:54:29.237

[66 rows x 2 columns]